In [1]:
from skimage import io
import numpy as np
from glob import glob
import sys
sys.path.append('../')
sys.path.append('/home/philipp/PycharmProjects/ps_course_projects-clb2_data_processing/src/')

from tracking import track_position
from napari_rf.features import FeatureCreator
from napari_rf.RF import RF
from joblib import load
from pathlib import Path
import re
import os
import matplotlib.pyplot as plt
from skimage.morphology import binary_erosion, binary_dilation
from tqdm import tqdm
import scipy.ndimage as ndi
import warnings
warnings.filterwarnings('ignore')

In [2]:
feature_creator = FeatureCreator()

In [3]:
def create_composite_features(feature_creator, *imgs):
    features = []
    for img in imgs:
        features.append(feature_creator.make_simple_features(img))
        
    return np.concatenate(features, axis=-1)

In [4]:
mappings = {
    '/media/philipp/WD1/clb2/processed_data/20230511_CLB2cellcycle_fovs002/':
    {
#         'wt': [5,6,7,8,9],
#         'zip': [17,18,19], #15, 16
#         'she2': [20,21,22,23,24],
#         'she3': [12,13,14], # 10,11, # CONTINUE HERE
    },
    '/media/philipp/WD1/clb2/processed_data/20230511_CLB2cellcycle_fovs003/':
    {
#         'wt': [10,11,12,13,14],
        'zip': [15,16], #,17,18,19
#         'she2': [5,6,7,8,9],
#         'she3': [20,21,22,23,24], # CONTINUE HERE
    },
    '/media/philipp/WD1/clb2/processed_data/20230512_CLB2cellcycle_fovs004/':
    {
#         'wt': [5,6,7,8,9],
        'zip': [15,16,17,18,19],
#         'she2': [20,21,22,23,24],
#         'she3': [10,11,12,13,14], # CONTINUE HERE
    },
}

In [5]:
classifiers = {
    'wt': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei.joblib')
        },
    'she2': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei.joblib')
        },
    'she3': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei.joblib')
        },
    'zip': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks_zip_mutant.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei_zip_mutant.joblib')
        }
}

In [6]:
for experiment, strains in mappings.items():
    print(experiment)
    for strain, positions in strains.items():
        for position in positions:
            path = f"{experiment}/position_{str(position).zfill(2)}/z_level_0/"
            os.makedirs(f"{path}/active_neck_proba", exist_ok=True)
            os.makedirs(f"{path}/inactive_neck_proba", exist_ok=True)
            os.makedirs(f"{path}/spindle_proba", exist_ok=True)
            os.makedirs(f"{path}/neck_segmentation", exist_ok=True)
            os.makedirs(f"{path}/neck_tracks", exist_ok=True)
            
            segmentation_imgs = sorted(glob(f"{path}/tracks_0/*"))
            
            for img_path in tqdm(segmentation_imgs):
                try:
                    img_name = f'{Path(img_path).stem}.tif'
                    channel_1 = io.imread(f"{path}/background_corrected_1/{img_name}")
                    channel_2 = io.imread(f"{path}/background_corrected_2/{img_name}")

                    features = create_composite_features(feature_creator, channel_1, channel_2)

                    neck_predictions = classifiers[strain]['neck_clf'].predict_segmenter(features)
                    spindle_predictions = classifiers[strain]['spindle_clf'].predict_segmenter(features)

                    io.imsave(f'{path}/active_neck_proba/{img_name}', neck_predictions[3].astype(np.float16))
                    io.imsave(f'{path}/inactive_neck_proba/{img_name}', neck_predictions[4].astype(np.float16))
                    io.imsave(f'{path}/spindle_proba/{img_name}', spindle_predictions[3].astype(np.float16))


                    q = np.argmax(neck_predictions, axis=0) == 3

                    for a in range(4):
                        q = binary_erosion(q)
                    for a in range(4):
                        q = binary_dilation(q)
                    labels, _ = ndi.label(q)

                    io.imsave(f"{path}/neck_segmentation/{Path(img_path).stem}.tif", labels.astype(np.uint16))
                except:
                    print(f'!FAILED {img_path}')
            track_position(f"{path}/neck_segmentation", f"{path}/neck_tracks")

/media/philipp/WD1/clb2/processed_data/20230511_CLB2cellcycle_fovs002/
/media/philipp/WD1/clb2/processed_data/20230511_CLB2cellcycle_fovs003/


100%|███████████████████████████████████████████| 73/73 [25:51<00:00, 21.26s/it]
73it [01:03,  1.15it/s]
72it [00:00, 3620.50it/s]
72it [00:00, 136.18it/s]
100%|███████████████████████████████████████████| 73/73 [36:54<00:00, 30.34s/it]
73it [01:28,  1.22s/it]
72it [00:01, 58.28it/s]
72it [00:01, 48.86it/s] 
100%|███████████████████████████████████████████| 73/73 [07:58<00:00,  6.56s/it]


/media/philipp/WD1/clb2/processed_data/20230512_CLB2cellcycle_fovs004/


 29%|████████████▎                              | 21/73 [09:17<22:25, 25.88s/it]

!FAILED /media/philipp/WD1/clb2/processed_data/20230512_CLB2cellcycle_fovs004//position_15/z_level_0//tracks_0/frame_020.tif


100%|███████████████████████████████████████████| 73/73 [28:14<00:00, 23.21s/it]
72it [01:26,  1.20s/it]
71it [00:00, 1877.75it/s]
71it [00:01, 53.84it/s] 
100%|███████████████████████████████████████████| 73/73 [28:01<00:00, 23.03s/it]
73it [00:42,  1.70it/s]
72it [00:00, 1414.05it/s]
72it [00:01, 44.37it/s] 
100%|███████████████████████████████████████████| 73/73 [29:58<00:00, 24.63s/it]
73it [02:17,  1.88s/it]
72it [00:00, 182.52it/s]
72it [00:03, 19.11it/s] 
 14%|█████▉                                     | 10/73 [05:18<24:58, 23.78s/it]

!FAILED /media/philipp/WD1/clb2/processed_data/20230512_CLB2cellcycle_fovs004//position_18/z_level_0//tracks_0/frame_009.tif


100%|███████████████████████████████████████████| 73/73 [30:46<00:00, 25.29s/it]
72it [01:56,  1.62s/it]
71it [00:00, 352.92it/s]
71it [00:05, 13.68it/s] 
100%|███████████████████████████████████████████| 73/73 [34:16<00:00, 28.18s/it]
73it [04:35,  3.77s/it]
72it [00:00, 89.96it/s] 
72it [00:07, 10.08it/s]
100%|███████████████████████████████████████████| 73/73 [04:45<00:00,  3.91s/it]


In [8]:
position

10

In [8]:
for path in glob(f'{root}/*'):
    position = re.findall('position_(\d+)', path)
    os.makedirs(f"{path}/active_neck_proba", exist_ok=True)
    os.makedirs(f"{path}/inactive_neck_proba", exist_ok=True)
    os.makedirs(f"{path}/spindle_proba", exist_ok=True)
    if position:
        strain = pos_to_strain.get(int(position[0]))
        
        segmentation_imgs = sorted(glob(f"{path}/tracks_0/*"))

        for img_path in tqdm(segmentation_imgs):
            img_name = f'{Path(img_path).stem}.tif'
            seg_img = io.imread(img_path)
            channel_1 = io.imread(f"{path}/background_corrected_1/{img_name}")
            channel_2 = io.imread(f"{path}/background_corrected_2/{img_name}")
            
            features = create_composite_features(feature_creator, channel_1, channel_2)
            
            neck_predictions = classifiers[strain]['neck_clf'].predict_segmenter(features)
            spindle_predictions = classifiers[strain]['spindle_clf'].predict_segmenter(features)
            
            io.imsave(f'{path}/active_neck_proba/{img_name}', neck_predictions[3].astype(np.float16))
            io.imsave(f'{path}/inactive_neck_proba/{img_name}', neck_predictions[4].astype(np.float16))
            io.imsave(f'{path}/spindle_proba/{img_name}', spindle_predictions[3].astype(np.float16))

100%|███████████████████████████████████████████| 68/68 [15:00<00:00, 13.25s/it]
